<a href="https://colab.research.google.com/github/dhruv9842smps-commits/Fake-News-Project/blob/main/Fake_News_Detection_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "5c5bcd21",
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import re\n",
    "import pandas as pd\n",
    "from nltk.corpus import stopwords\n",
    "from nltk.stem.porter import PorterStemmer\n",
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.metrics import accuracy_score"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "e205be1f",
   "metadata": {},
   "outputs": [],
   "source": [
    "news_df = pd.read_csv('train.csv')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "d64ec0d0",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>id</th>\n",
       "      <th>title</th>\n",
       "      <th>author</th>\n",
       "      <th>text</th>\n",
       "      <th>label</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>0</td>\n",
       "      <td>House Dem Aide: We Didn’t Even See Comey’s Let...</td>\n",
       "      <td>Darrell Lucus</td>\n",
       "      <td>House Dem Aide: We Didn’t Even See Comey’s Let...</td>\n",
       "      <td>1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>1</td>\n",
       "      <td>FLYNN: Hillary Clinton, Big Woman on Campus - ...</td>\n",
       "      <td>Daniel J. Flynn</td>\n",
       "      <td>Ever get the feeling your life circles the rou...</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>2</td>\n",
       "      <td>Why the Truth Might Get You Fired</td>\n",
       "      <td>Consortiumnews.com</td>\n",
       "      <td>Why the Truth Might Get You Fired October 29, ...</td>\n",
       "      <td>1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>3</td>\n",
       "      <td>15 Civilians Killed In Single US Airstrike Hav...</td>\n",
       "      <td>Jessica Purkiss</td>\n",
       "      <td>Videos 15 Civilians Killed In Single US Airstr...</td>\n",
       "      <td>1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>4</td>\n",
       "      <td>Iranian woman jailed for fictional unpublished...</td>\n",
       "      <td>Howard Portnoy</td>\n",
       "      <td>Print \\nAn Iranian woman has been sentenced to...</td>\n",
       "      <td>1</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "   id                                              title              author  \\\n",
       "0   0  House Dem Aide: We Didn’t Even See Comey’s Let...       Darrell Lucus   \n",
       "1   1  FLYNN: Hillary Clinton, Big Woman on Campus - ...     Daniel J. Flynn   \n",
       "2   2                  Why the Truth Might Get You Fired  Consortiumnews.com   \n",
       "3   3  15 Civilians Killed In Single US Airstrike Hav...     Jessica Purkiss   \n",
       "4   4  Iranian woman jailed for fictional unpublished...      Howard Portnoy   \n",
       "\n",
       "                                                text  label  \n",
       "0  House Dem Aide: We Didn’t Even See Comey’s Let...      1  \n",
       "1  Ever get the feeling your life circles the rou...      0  \n",
       "2  Why the Truth Might Get You Fired October 29, ...      1  \n",
       "3  Videos 15 Civilians Killed In Single US Airstr...      1  \n",
       "4  Print \\nAn Iranian woman has been sentenced to...      1  "
      ]
     },
     "execution_count": 3,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5a79c331",
   "metadata": {},
   "source": [
    "# About the Dataset:\n",
    "\n",
    "id: unique id for a news article                                     \n",
    "title: the title of a news article                                  \n",
    "author: author of the news article                                    \n",
    "text: the text of the article; could be incomplete                                        \n",
    "label: a label that marks whether the news article is real or fake:                              \n",
    "    1: Fake news                                                 \n",
    "    0: real News                                        "
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2148a04e",
   "metadata": {},
   "source": [
    "# 1 Preprocessing "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "80b053d5",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "id           0\n",
       "title      558\n",
       "author    1957\n",
       "text        39\n",
       "label        0\n",
       "dtype: int64"
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df.isnull().sum()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "5ab108cd",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "(20800, 5)"
      ]
     },
     "execution_count": 5,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df.shape"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "286eb295",
   "metadata": {},
   "outputs": [],
   "source": [
    "news_df = news_df.fillna(' ')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "94cb67ce",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "id        0\n",
       "title     0\n",
       "author    0\n",
       "text      0\n",
       "label     0\n",
       "dtype: int64"
      ]
     },
     "execution_count": 7,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df.isnull().sum()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "3d787683",
   "metadata": {},
   "outputs": [],
   "source": [
    "news_df['content'] = news_df['author']+' '+news_df['title']"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "e399192b",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>id</th>\n",
       "      <th>title</th>\n",
       "      <th>author</th>\n",
       "      <th>text</th>\n",
       "      <th>label</th>\n",
       "      <th>content</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>0</td>\n",
       "      <td>House Dem Aide: We Didn’t Even See Comey’s Let...</td>\n",
       "      <td>Darrell Lucus</td>\n",
       "      <td>House Dem Aide: We Didn’t Even See Comey’s Let...</td>\n",
       "      <td>1</td>\n",
       "      <td>Darrell Lucus House Dem Aide: We Didn’t Even S...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>1</td>\n",
       "      <td>FLYNN: Hillary Clinton, Big Woman on Campus - ...</td>\n",
       "      <td>Daniel J. Flynn</td>\n",
       "      <td>Ever get the feeling your life circles the rou...</td>\n",
       "      <td>0</td>\n",
       "      <td>Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>2</td>\n",
       "      <td>Why the Truth Might Get You Fired</td>\n",
       "      <td>Consortiumnews.com</td>\n",
       "      <td>Why the Truth Might Get You Fired October 29, ...</td>\n",
       "      <td>1</td>\n",
       "      <td>Consortiumnews.com Why the Truth Might Get You...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>3</th>\n",
       "      <td>3</td>\n",
       "      <td>15 Civilians Killed In Single US Airstrike Hav...</td>\n",
       "      <td>Jessica Purkiss</td>\n",
       "      <td>Videos 15 Civilians Killed In Single US Airstr...</td>\n",
       "      <td>1</td>\n",
       "      <td>Jessica Purkiss 15 Civilians Killed In Single ...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>4</th>\n",
       "      <td>4</td>\n",
       "      <td>Iranian woman jailed for fictional unpublished...</td>\n",
       "      <td>Howard Portnoy</td>\n",
       "      <td>Print \\nAn Iranian woman has been sentenced to...</td>\n",
       "      <td>1</td>\n",
       "      <td>Howard Portnoy Iranian woman jailed for fictio...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>...</th>\n",
       "      <td>...</td>\n",
       "      <td>...</td>\n",
       "      <td>...</td>\n",
       "      <td>...</td>\n",
       "      <td>...</td>\n",
       "      <td>...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>20795</th>\n",
       "      <td>20795</td>\n",
       "      <td>Rapper T.I.: Trump a ’Poster Child For White S...</td>\n",
       "      <td>Jerome Hudson</td>\n",
       "      <td>Rapper T. I. unloaded on black celebrities who...</td>\n",
       "      <td>0</td>\n",
       "      <td>Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>20796</th>\n",
       "      <td>20796</td>\n",
       "      <td>N.F.L. Playoffs: Schedule, Matchups and Odds -...</td>\n",
       "      <td>Benjamin Hoffman</td>\n",
       "      <td>When the Green Bay Packers lost to the Washing...</td>\n",
       "      <td>0</td>\n",
       "      <td>Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>20797</th>\n",
       "      <td>20797</td>\n",
       "      <td>Macy’s Is Said to Receive Takeover Approach by...</td>\n",
       "      <td>Michael J. de la Merced and Rachel Abrams</td>\n",
       "      <td>The Macy’s of today grew from the union of sev...</td>\n",
       "      <td>0</td>\n",
       "      <td>Michael J. de la Merced and Rachel Abrams Macy...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>20798</th>\n",
       "      <td>20798</td>\n",
       "      <td>NATO, Russia To Hold Parallel Exercises In Bal...</td>\n",
       "      <td>Alex Ansary</td>\n",
       "      <td>NATO, Russia To Hold Parallel Exercises In Bal...</td>\n",
       "      <td>1</td>\n",
       "      <td>Alex Ansary NATO, Russia To Hold Parallel Exer...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>20799</th>\n",
       "      <td>20799</td>\n",
       "      <td>What Keeps the F-35 Alive</td>\n",
       "      <td>David Swanson</td>\n",
       "      <td>David Swanson is an author, activist, journa...</td>\n",
       "      <td>1</td>\n",
       "      <td>David Swanson What Keeps the F-35 Alive</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "<p>20800 rows × 6 columns</p>\n",
       "</div>"
      ],
      "text/plain": [
       "          id                                              title  \\\n",
       "0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   \n",
       "1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   \n",
       "2          2                  Why the Truth Might Get You Fired   \n",
       "3          3  15 Civilians Killed In Single US Airstrike Hav...   \n",
       "4          4  Iranian woman jailed for fictional unpublished...   \n",
       "...      ...                                                ...   \n",
       "20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   \n",
       "20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   \n",
       "20797  20797  Macy’s Is Said to Receive Takeover Approach by...   \n",
       "20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   \n",
       "20799  20799                          What Keeps the F-35 Alive   \n",
       "\n",
       "                                          author  \\\n",
       "0                                  Darrell Lucus   \n",
       "1                                Daniel J. Flynn   \n",
       "2                             Consortiumnews.com   \n",
       "3                                Jessica Purkiss   \n",
       "4                                 Howard Portnoy   \n",
       "...                                          ...   \n",
       "20795                              Jerome Hudson   \n",
       "20796                           Benjamin Hoffman   \n",
       "20797  Michael J. de la Merced and Rachel Abrams   \n",
       "20798                                Alex Ansary   \n",
       "20799                              David Swanson   \n",
       "\n",
       "                                                    text  label  \\\n",
       "0      House Dem Aide: We Didn’t Even See Comey’s Let...      1   \n",
       "1      Ever get the feeling your life circles the rou...      0   \n",
       "2      Why the Truth Might Get You Fired October 29, ...      1   \n",
       "3      Videos 15 Civilians Killed In Single US Airstr...      1   \n",
       "4      Print \\nAn Iranian woman has been sentenced to...      1   \n",
       "...                                                  ...    ...   \n",
       "20795  Rapper T. I. unloaded on black celebrities who...      0   \n",
       "20796  When the Green Bay Packers lost to the Washing...      0   \n",
       "20797  The Macy’s of today grew from the union of sev...      0   \n",
       "20798  NATO, Russia To Hold Parallel Exercises In Bal...      1   \n",
       "20799    David Swanson is an author, activist, journa...      1   \n",
       "\n",
       "                                                 content  \n",
       "0      Darrell Lucus House Dem Aide: We Didn’t Even S...  \n",
       "1      Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...  \n",
       "2      Consortiumnews.com Why the Truth Might Get You...  \n",
       "3      Jessica Purkiss 15 Civilians Killed In Single ...  \n",
       "4      Howard Portnoy Iranian woman jailed for fictio...  \n",
       "...                                                  ...  \n",
       "20795  Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...  \n",
       "20796  Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...  \n",
       "20797  Michael J. de la Merced and Rachel Abrams Macy...  \n",
       "20798  Alex Ansary NATO, Russia To Hold Parallel Exer...  \n",
       "20799            David Swanson What Keeps the F-35 Alive  \n",
       "\n",
       "[20800 rows x 6 columns]"
      ]
     },
     "execution_count": 9,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a5cb5a30",
   "metadata": {},
   "source": [
    "# separating the data & label"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "bc35715c",
   "metadata": {},
   "outputs": [],
   "source": [
    "X = news_df.drop('label',axis=1)\n",
    "y = news_df['label']"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "9196cb1f",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "          id                                              title  \\\n",
      "0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   \n",
      "1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   \n",
      "2          2                  Why the Truth Might Get You Fired   \n",
      "3          3  15 Civilians Killed In Single US Airstrike Hav...   \n",
      "4          4  Iranian woman jailed for fictional unpublished...   \n",
      "...      ...                                                ...   \n",
      "20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   \n",
      "20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   \n",
      "20797  20797  Macy’s Is Said to Receive Takeover Approach by...   \n",
      "20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   \n",
      "20799  20799                          What Keeps the F-35 Alive   \n",
      "\n",
      "                                          author  \\\n",
      "0                                  Darrell Lucus   \n",
      "1                                Daniel J. Flynn   \n",
      "2                             Consortiumnews.com   \n",
      "3                                Jessica Purkiss   \n",
      "4                                 Howard Portnoy   \n",
      "...                                          ...   \n",
      "20795                              Jerome Hudson   \n",
      "20796                           Benjamin Hoffman   \n",
      "20797  Michael J. de la Merced and Rachel Abrams   \n",
      "20798                                Alex Ansary   \n",
      "20799                              David Swanson   \n",
      "\n",
      "                                                    text  \\\n",
      "0      House Dem Aide: We Didn’t Even See Comey’s Let...   \n",
      "1      Ever get the feeling your life circles the rou...   \n",
      "2      Why the Truth Might Get You Fired October 29, ...   \n",
      "3      Videos 15 Civilians Killed In Single US Airstr...   \n",
      "4      Print \\nAn Iranian woman has been sentenced to...   \n",
      "...                                                  ...   \n",
      "20795  Rapper T. I. unloaded on black celebrities who...   \n",
      "20796  When the Green Bay Packers lost to the Washing...   \n",
      "20797  The Macy’s of today grew from the union of sev...   \n",
      "20798  NATO, Russia To Hold Parallel Exercises In Bal...   \n",
      "20799    David Swanson is an author, activist, journa...   \n",
      "\n",
      "                                                 content  \n",
      "0      Darrell Lucus House Dem Aide: We Didn’t Even S...  \n",
      "1      Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...  \n",
      "2      Consortiumnews.com Why the Truth Might Get You...  \n",
      "3      Jessica Purkiss 15 Civilians Killed In Single ...  \n",
      "4      Howard Portnoy Iranian woman jailed for fictio...  \n",
      "...                                                  ...  \n",
      "20795  Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...  \n",
      "20796  Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...  \n",
      "20797  Michael J. de la Merced and Rachel Abrams Macy...  \n",
      "20798  Alex Ansary NATO, Russia To Hold Parallel Exer...  \n",
      "20799            David Swanson What Keeps the F-35 Alive  \n",
      "\n",
      "[20800 rows x 5 columns]\n"
     ]
    }
   ],
   "source": [
    "print(X)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "a3579e9f",
   "metadata": {},
   "source": [
    "# Stemming:\n",
    "\n",
    "Stemming is the process of reducing a word to its Root word\n",
    "\n",
    "example: hung         hanged        hanging ======hang"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "c4f4cba0",
   "metadata": {},
   "source": [
    "# Steps:\n",
    "lower case                 \n",
    "splitting                             \n",
    "removing stopwords                              \n",
    "stemming                                   "
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "c8ef2a22",
   "metadata": {},
   "outputs": [],
   "source": [
    "ps = PorterStemmer()\n",
    "def stemming(content):\n",
    "    stemmed_content = re.sub('[^a-zA-Z]',' ',content)\n",
    "    stemmed_content = stemmed_content.lower()\n",
    "    stemmed_content = stemmed_content.split()\n",
    "    stemmed_content = [ps.stem(word) for word in stemmed_content if not word in stopwords.words('english')]\n",
    "    stemmed_content = ' '.join(stemmed_content)\n",
    "    return stemmed_content"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "1f1fd73c",
   "metadata": {},
   "outputs": [],
   "source": [
    "news_df['content'] = news_df['content'].apply(stemming)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "70e553da",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "0        darrel lucu hous dem aid even see comey letter...\n",
       "1        daniel j flynn flynn hillari clinton big woman...\n",
       "2                   consortiumnew com truth might get fire\n",
       "3        jessica purkiss civilian kill singl us airstri...\n",
       "4        howard portnoy iranian woman jail fiction unpu...\n",
       "                               ...                        \n",
       "20795    jerom hudson rapper trump poster child white s...\n",
       "20796    benjamin hoffman n f l playoff schedul matchup...\n",
       "20797    michael j de la merc rachel abram maci said re...\n",
       "20798    alex ansari nato russia hold parallel exercis ...\n",
       "20799                            david swanson keep f aliv\n",
       "Name: content, Length: 20800, dtype: object"
      ]
     },
     "execution_count": 14,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df['content']"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2a0f3325",
   "metadata": {},
   "source": [
    "# separating the data and label\n"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "d9fe53c2",
   "metadata": {},
   "outputs": [],
   "source": [
    "X = news_df['content'].values\n",
    "y = news_df['label'].values"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f93b9984",
   "metadata": {},
   "source": [
    "# converting the textual data to numerical data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "1abf344d",
   "metadata": {},
   "outputs": [],
   "source": [
    "vector = TfidfVectorizer()\n",
    "vector.fit(X)\n",
    "X = vector.transform(X)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "e60f95d1",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "  (0, 15686)\t0.28485063562728646\n",
      "  (0, 13473)\t0.2565896679337957\n",
      "  (0, 8909)\t0.3635963806326075\n",
      "  (0, 8630)\t0.29212514087043684\n",
      "  (0, 7692)\t0.24785219520671603\n",
      "  (0, 7005)\t0.21874169089359144\n",
      "  (0, 4973)\t0.233316966909351\n",
      "  (0, 3792)\t0.2705332480845492\n",
      "  (0, 3600)\t0.3598939188262559\n",
      "  (0, 2959)\t0.2468450128533713\n",
      "  (0, 2483)\t0.3676519686797209\n",
      "  (0, 267)\t0.27010124977708766\n",
      "  (1, 16799)\t0.30071745655510157\n",
      "  (1, 6816)\t0.1904660198296849\n",
      "  (1, 5503)\t0.7143299355715573\n",
      "  (1, 3568)\t0.26373768806048464\n",
      "  (1, 2813)\t0.19094574062359204\n",
      "  (1, 2223)\t0.3827320386859759\n",
      "  (1, 1894)\t0.15521974226349364\n",
      "  (1, 1497)\t0.2939891562094648\n",
      "  (2, 15611)\t0.41544962664721613\n",
      "  (2, 9620)\t0.49351492943649944\n",
      "  (2, 5968)\t0.3474613386728292\n",
      "  (2, 5389)\t0.3866530551182615\n",
      "  (2, 3103)\t0.46097489583229645\n",
      "  :\t:\n",
      "  (20797, 13122)\t0.2482526352197606\n",
      "  (20797, 12344)\t0.27263457663336677\n",
      "  (20797, 12138)\t0.24778257724396507\n",
      "  (20797, 10306)\t0.08038079000566466\n",
      "  (20797, 9588)\t0.174553480255222\n",
      "  (20797, 9518)\t0.2954204003420313\n",
      "  (20797, 8988)\t0.36160868928090795\n",
      "  (20797, 8364)\t0.22322585870464118\n",
      "  (20797, 7042)\t0.21799048897828688\n",
      "  (20797, 3643)\t0.21155500613623743\n",
      "  (20797, 1287)\t0.33538056804139865\n",
      "  (20797, 699)\t0.30685846079762347\n",
      "  (20797, 43)\t0.29710241860700626\n",
      "  (20798, 13046)\t0.22363267488270608\n",
      "  (20798, 11052)\t0.4460515589182236\n",
      "  (20798, 10177)\t0.3192496370187028\n",
      "  (20798, 6889)\t0.32496285694299426\n",
      "  (20798, 5032)\t0.4083701450239529\n",
      "  (20798, 1125)\t0.4460515589182236\n",
      "  (20798, 588)\t0.3112141524638974\n",
      "  (20798, 350)\t0.28446937819072576\n",
      "  (20799, 14852)\t0.5677577267055112\n",
      "  (20799, 8036)\t0.45983893273780013\n",
      "  (20799, 3623)\t0.37927626273066584\n",
      "  (20799, 377)\t0.5677577267055112\n"
     ]
    }
   ],
   "source": [
    "print(X)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7a4c1ab4",
   "metadata": {},
   "source": [
    "# Splitting the dataset to training & test data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "12a0d4eb",
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size = 0.2, stratify=y, random_state=2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 25,
   "id": "6402a62e",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "(16640, 17128)"
      ]
     },
     "execution_count": 25,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "X_train.shape"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "83960b57",
   "metadata": {},
   "outputs": [],
   "source": []
  },
  {
   "cell_type": "markdown",
   "id": "7a41062a",
   "metadata": {},
   "source": [
    "# Training the Model: Logistic Regression"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "f22c5fa7",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/html": [
       "<style>#sk-container-id-1 {color: black;background-color: white;}#sk-container-id-1 pre{padding: 0;}#sk-container-id-1 div.sk-toggleable {background-color: white;}#sk-container-id-1 label.sk-toggleable__label {cursor: pointer;display: block;width: 100%;margin-bottom: 0;padding: 0.3em;box-sizing: border-box;text-align: center;}#sk-container-id-1 label.sk-toggleable__label-arrow:before {content: \"▸\";float: left;margin-right: 0.25em;color: #696969;}#sk-container-id-1 label.sk-toggleable__label-arrow:hover:before {color: black;}#sk-container-id-1 div.sk-estimator:hover label.sk-toggleable__label-arrow:before {color: black;}#sk-container-id-1 div.sk-toggleable__content {max-height: 0;max-width: 0;overflow: hidden;text-align: left;background-color: #f0f8ff;}#sk-container-id-1 div.sk-toggleable__content pre {margin: 0.2em;color: black;border-radius: 0.25em;background-color: #f0f8ff;}#sk-container-id-1 input.sk-toggleable__control:checked~div.sk-toggleable__content {max-height: 200px;max-width: 100%;overflow: auto;}#sk-container-id-1 input.sk-toggleable__control:checked~label.sk-toggleable__label-arrow:before {content: \"▾\";}#sk-container-id-1 div.sk-estimator input.sk-toggleable__control:checked~label.sk-toggleable__label {background-color: #d4ebff;}#sk-container-id-1 div.sk-label input.sk-toggleable__control:checked~label.sk-toggleable__label {background-color: #d4ebff;}#sk-container-id-1 input.sk-hidden--visually {border: 0;clip: rect(1px 1px 1px 1px);clip: rect(1px, 1px, 1px, 1px);height: 1px;margin: -1px;overflow: hidden;padding: 0;position: absolute;width: 1px;}#sk-container-id-1 div.sk-estimator {font-family: monospace;background-color: #f0f8ff;border: 1px dotted black;border-radius: 0.25em;box-sizing: border-box;margin-bottom: 0.5em;}#sk-container-id-1 div.sk-estimator:hover {background-color: #d4ebff;}#sk-container-id-1 div.sk-parallel-item::after {content: \"\";width: 100%;border-bottom: 1px solid gray;flex-grow: 1;}#sk-container-id-1 div.sk-label:hover label.sk-toggleable__label {background-color: #d4ebff;}#sk-container-id-1 div.sk-serial::before {content: \"\";position: absolute;border-left: 1px solid gray;box-sizing: border-box;top: 0;bottom: 0;left: 50%;z-index: 0;}#sk-container-id-1 div.sk-serial {display: flex;flex-direction: column;align-items: center;background-color: white;padding-right: 0.2em;padding-left: 0.2em;position: relative;}#sk-container-id-1 div.sk-item {position: relative;z-index: 1;}#sk-container-id-1 div.sk-parallel {display: flex;align-items: stretch;justify-content: center;background-color: white;position: relative;}#sk-container-id-1 div.sk-item::before, #sk-container-id-1 div.sk-parallel-item::before {content: \"\";position: absolute;border-left: 1px solid gray;box-sizing: border-box;top: 0;bottom: 0;left: 50%;z-index: -1;}#sk-container-id-1 div.sk-parallel-item {display: flex;flex-direction: column;z-index: 1;position: relative;background-color: white;}#sk-container-id-1 div.sk-parallel-item:first-child::after {align-self: flex-end;width: 50%;}#sk-container-id-1 div.sk-parallel-item:last-child::after {align-self: flex-start;width: 50%;}#sk-container-id-1 div.sk-parallel-item:only-child::after {width: 0;}#sk-container-id-1 div.sk-dashed-wrapped {border: 1px dashed gray;margin: 0 0.4em 0.5em 0.4em;box-sizing: border-box;padding-bottom: 0.4em;background-color: white;}#sk-container-id-1 div.sk-label label {font-family: monospace;font-weight: bold;display: inline-block;line-height: 1.2em;}#sk-container-id-1 div.sk-label-container {text-align: center;}#sk-container-id-1 div.sk-container {/* jupyter's `normalize.less` sets `[hidden] { display: none; }` but bootstrap.min.css set `[hidden] { display: none !important; }` so we also need the `!important` here to be able to override the default hidden behavior on the sphinx rendered scikit-learn.org. See: https://github.com/scikit-learn/scikit-learn/issues/21755 */display: inline-block !important;position: relative;}#sk-container-id-1 div.sk-text-repr-fallback {display: none;}</style><div id=\"sk-container-id-1\" class=\"sk-top-container\"><div class=\"sk-text-repr-fallback\"><pre>LogisticRegression()</pre><b>In a Jupyter environment, please rerun this cell to show the HTML representation or trust the notebook. <br />On GitHub, the HTML representation is unable to render, please try loading this page with nbviewer.org.</b></div><div class=\"sk-container\" hidden><div class=\"sk-item\"><div class=\"sk-estimator sk-toggleable\"><input class=\"sk-toggleable__control sk-hidden--visually\" id=\"sk-estimator-id-1\" type=\"checkbox\" checked><label for=\"sk-estimator-id-1\" class=\"sk-toggleable__label sk-toggleable__label-arrow\">LogisticRegression</label><div class=\"sk-toggleable__content\"><pre>LogisticRegression()</pre></div></div></div></div></div>"
      ],
      "text/plain": [
       "LogisticRegression()"
      ]
     },
     "execution_count": 19,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "model = LogisticRegression()\n",
    "model.fit(X_train,Y_train)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "7800d38d",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "0.9865985576923076\n"
     ]
    }
   ],
   "source": [
    "# on training set\n",
    "train_y_pred = model.predict(X_train)\n",
    "print(accuracy_score(train_y_pred,Y_train))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "bc974492",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "0.9790865384615385\n"
     ]
    }
   ],
   "source": [
    "# on testing set\n",
    "testing_y_pred = model.predict(X_test)\n",
    "print(accuracy_score(testing_y_pred,Y_test))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e316cc88",
   "metadata": {},
   "source": [
    "# Detection System"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 22,
   "id": "882436d7",
   "metadata": {},
   "outputs": [],
   "source": [
    "input_data = X_test[10]\n",
    "prediction = model.predict(input_data)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 23,
   "id": "ecad7136",
   "metadata": {},
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "The News Is Real\n"
     ]
    }
   ],
   "source": [
    "if prediction[0] == 0:\n",
    "    print('The News Is Real')\n",
    "else:\n",
    "    print('The News is Fake')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 24,
   "id": "36fba43f",
   "metadata": {},
   "outputs": [
    {
     "data": {
      "text/plain": [
       "'consortiumnew com truth might get fire'"
      ]
     },
     "execution_count": 24,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "news_df['content'][2]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": None,
   "id": "33412466",
   "metadata": {},
   "outputs": [],
   "source": []
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.10.6"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'code',
   'execution_count': 1,
   'id': '5c5bcd21',
   'metadata': {},
   'outputs': [],
   'source': ['import numpy as np\n',
    'import re\n',
    'import pandas as pd\n',
    'from nltk.corpus import stopwords\n',
    'from nltk.stem.porter import PorterStemmer\n',
    'from sklearn.feature_extraction.text import TfidfVectorizer\n',
    'from sklearn.model_selection import train_test_split\n',
    'from sklearn.linear_model import LogisticRegression\n',
    'from sklearn.metrics import accuracy_score']},
  {'cell_type': 'code',
   'execution_count': 2,
   'id': 'e205be1f',
   'metadata': {},
   'outputs': [],
   'source': ["news_df = pd.read_csv('train.csv')"]},
  {'cell_type': 'code',
   'execution_count': 3,
   'id': 'd64ec0d0',
   'metadata': {},
   'outputs': [{'data': {'text/html': ['<div>\n',
       '<style scoped>\n',
       '    .dataframe tbody tr th:only-of-type {\n',
       '        vertical-align: middle;\n',
       '    }\n',
       '\n',
   